In [221]:
import psycopg2
import streamlit as st


In [222]:
def get_connection():
    try:
        conn = psycopg2.connect(
            host="localhost",
            database="Cricbuzz_Capstone",
            user="postgres",
            password="280402",
            port="5432"
        )
        return conn
    except Exception as e:
        st.error(f"PostgreSQL connection failed: {e}")
        return None

In [223]:
import http.client

conn = http.client.HTTPSConnection("cricbuzz-cricket.p.rapidapi.com")

headers = {
    'x-rapidapi-key': "d1d1206e78mshb418f1da663697ap109e25jsn86275d1f5944",
    'x-rapidapi-host': "cricbuzz-cricket.p.rapidapi.com"
}

conn.request("GET", "/matches/v1/live", headers=headers)

res = conn.getresponse()
data = res.read()

print(data.decode("utf-8"))

{"message":"You have exceeded the MONTHLY quota for Requests on your current plan, BASIC. Upgrade your plan at https:\/\/rapidapi.com\/cricketapilive\/api\/cricbuzz-cricket"}


In [263]:
import http.client

conn = http.client.HTTPSConnection("cricbuzz-cricket.p.rapidapi.com")

headers = {
    'x-rapidapi-key': "d1d1206e78mshb418f1da663697ap109e25jsn86275d1f5944",
    'x-rapidapi-host': "cricbuzz-cricket.p.rapidapi.com"
}

conn.request("GET", "/stats/v1/topstats/0?statsType=mostWickets", headers=headers)

res = conn.getresponse()
data = res.read()

print(data.decode("utf-8"))

{"message":"You have exceeded the MONTHLY quota for Requests on your current plan, BASIC. Upgrade your plan at https:\/\/rapidapi.com\/cricketapilive\/api\/cricbuzz-cricket"}


In [225]:
import http.client

conn = http.client.HTTPSConnection("cricbuzz-cricket.p.rapidapi.com")

headers = {
    'x-rapidapi-key': "d1d1206e78mshb418f1da663697ap109e25jsn86275d1f5944",
    'x-rapidapi-host': "cricbuzz-cricket.p.rapidapi.com",
    'Content-Type': "application/json"
}

conn.request("GET", "/teams/v1/2/players", headers=headers)

res = conn.getresponse()
data = res.read()

print(data.decode("utf-8"))

{"message":"You have exceeded the MONTHLY quota for Requests on your current plan, BASIC. Upgrade your plan at https:\/\/rapidapi.com\/cricketapilive\/api\/cricbuzz-cricket"}


International Cricket Teams


In [268]:
import http.client
import json

def fetch_and_insert_teams():
    conn_db = get_connection()
    if not conn_db:
        return

    cursor = conn_db.cursor()

    # API connection
    conn = http.client.HTTPSConnection("cricbuzz-cricket.p.rapidapi.com")

    headers = {
         'x-rapidapi-key': "274cf7b009msh198245adc41e439p1f3633jsna216dc087842",
    'x-rapidapi-host': "cricbuzz-cricket.p.rapidapi.com",
    }

    conn.request("GET", "/teams/v1/international", headers=headers)

    res = conn.getresponse()
    data = json.loads(res.read().decode("utf-8"))

    current_category = None

    for item in data["list"]:

        # Handle category rows like "Test Teams"
        if "teamId" not in item:
            current_category = item.get("teamName")
            continue

        team_id = item["teamId"]
        team_name = item["teamName"]
        short_name = item.get("teamSName")

        try:
            cursor.execute("""
                INSERT INTO teams (team_id, team_name, short_name, category)
                VALUES (%s, %s, %s, %s)
                ON CONFLICT (team_id) DO NOTHING;
            """, (team_id, team_name, short_name, current_category))

        except Exception as e:
            print(f"Error inserting team {team_name}: {e}")

    conn_db.commit()
    cursor.close()
    conn_db.close()

    print("✅ Teams inserted successfully!")


# run
fetch_and_insert_teams()

✅ Teams inserted successfully!


In [271]:
import http.client
import json
import time
import psycopg2


# ✅ Your DB connection
def get_connection():
    try:
        conn = psycopg2.connect(
            host="localhost",
            database="Cricbuzz_Capstone",
            user="postgres",
            password="280402",
            port="5432"
        )
        return conn
    except Exception as e:
        print(f"PostgreSQL connection failed: {e}")
        return None


def fetch_and_insert_players():
    conn_db = get_connection()
    if not conn_db:
        return

    cursor = conn_db.cursor()

    # ✅ Get all team_ids
    cursor.execute("SELECT team_id FROM teams;")
    teams = cursor.fetchall()

    headers = {
           'x-rapidapi-key': "274cf7b009msh198245adc41e439p1f3633jsna216dc087842",
    'x-rapidapi-host': "cricbuzz-cricket.p.rapidapi.com",
    }

    for (team_id,) in teams:
        print(f"\n📡 Fetching players for team {team_id}...")

        try:
            # ✅ FIX: new connection per request
            conn_api = http.client.HTTPSConnection("cricbuzz-cricket.p.rapidapi.com")

            conn_api.request("GET", f"/teams/v1/{team_id}/players", headers=headers)
            res = conn_api.getresponse()

            # ❌ No data case (normal)
            if res.status == 204:
                print(f"⚠️ No players for team {team_id} (204)")
                continue

            # ❌ Any other failure
            if res.status != 200:
                print(f"❌ Failed for team {team_id}, status: {res.status}")
                continue

            data = json.loads(res.read().decode("utf-8"))

            current_role = None

            for item in data.get("player", []):

                # ✅ Handle role headers
                if "id" not in item:
                    current_role = item.get("name")
                    continue

                player_id = int(item["id"])
                player_name = item["name"]
                batting_style = item.get("battingStyle")
                bowling_style = item.get("bowlingStyle")

                cursor.execute("""
                    INSERT INTO players (
                        player_id,
                        player_name,
                        team_id,
                        batting_style,
                        bowling_style,
                        role
                    )
                    VALUES (%s, %s, %s, %s, %s, %s)
                    ON CONFLICT (player_id) DO NOTHING;
                """, (
                    player_id,
                    player_name,
                    team_id,
                    batting_style,
                    bowling_style,
                    current_role
                ))

            conn_db.commit()
            print(f"✅ Inserted players for team {team_id}")

            # 🔥 IMPORTANT → avoid rate limit
            time.sleep(1)

        except Exception as e:
            print(f"❌ Error for team {team_id}: {e}")

    cursor.close()
    conn_db.close()

    print("\n🎉 All players inserted successfully!")


# ▶️ RUN
fetch_and_insert_players()


📡 Fetching players for team 2...
✅ Inserted players for team 2

📡 Fetching players for team 96...
✅ Inserted players for team 96

📡 Fetching players for team 27...
✅ Inserted players for team 27

📡 Fetching players for team 3...
✅ Inserted players for team 3

📡 Fetching players for team 4...
✅ Inserted players for team 4

📡 Fetching players for team 5...
✅ Inserted players for team 5

📡 Fetching players for team 6...
✅ Inserted players for team 6

📡 Fetching players for team 9...
✅ Inserted players for team 9

📡 Fetching players for team 10...
✅ Inserted players for team 10

📡 Fetching players for team 11...
✅ Inserted players for team 11

📡 Fetching players for team 12...
✅ Inserted players for team 12

📡 Fetching players for team 13...
✅ Inserted players for team 13

📡 Fetching players for team 71...
⚠️ No players for team 71 (204)

📡 Fetching players for team 72...
✅ Inserted players for team 72

📡 Fetching players for team 77...
⚠️ No players for team 77 (204)

📡 Fetching players 

In [277]:
import http.client
import json

def fetch_and_insert_matches():
    conn_db = get_connection()
    if not conn_db:
        return

    cursor = conn_db.cursor()

    headers = {
         'x-rapidapi-key': "274cf7b009msh198245adc41e439p1f3633jsna216dc087842",
    'x-rapidapi-host': "cricbuzz-cricket.p.rapidapi.com",
    }

    conn_api = http.client.HTTPSConnection("cricbuzz-cricket.p.rapidapi.com")
    conn_api.request("GET", "/matches/v1/recent", headers=headers)
    res = conn_api.getresponse()

    data = json.loads(res.read().decode("utf-8"))

    for type_match in data.get("typeMatches", []):
        for series in type_match.get("seriesMatches", []):

            # ✅ skip ads
            if "seriesAdWrapper" not in series:
                continue

            series_data = series["seriesAdWrapper"]
            series_id = series_data.get("seriesId")

            for match in series_data.get("matches", []):

                match_info = match.get("matchInfo", {})
                match_id = match_info.get("matchId")

                team1 = match_info.get("team1", {})
                team2 = match_info.get("team2", {})

                team1_id = team1.get("teamId")
                team2_id = team2.get("teamId")

                team1_name = team1.get("teamName")
                team2_name = team2.get("teamName")

                match_desc = match_info.get("matchDesc")
                match_format = match_info.get("matchFormat")
                status = match_info.get("status")
                state = match_info.get("state")

                start_date = match_info.get("startDate")
                end_date = match_info.get("endDate")

                venue = match_info.get("venueInfo", {}).get("ground")

                try:
                    # 🔥 AUTO-INSERT missing teams (CRITICAL FIX)
                    cursor.execute("""
                        INSERT INTO teams (team_id, team_name)
                        VALUES (%s, %s)
                        ON CONFLICT (team_id) DO NOTHING;
                    """, (team1_id, team1_name))

                    cursor.execute("""
                        INSERT INTO teams (team_id, team_name)
                        VALUES (%s, %s)
                        ON CONFLICT (team_id) DO NOTHING;
                    """, (team2_id, team2_name))

                    # 🔥 derive winner
                    winner_id = None
                    if state == "Complete" and status:
                        if team1_name and team1_name in status:
                            winner_id = team1_id
                        elif team2_name and team2_name in status:
                            winner_id = team2_id

                    # ✅ insert match
                    cursor.execute("""
                        INSERT INTO matches (
                            match_id, series_id, match_desc, match_format,
                            start_date, end_date, status,
                            team1_id, team2_id, winner_team_id, venue
                        )
                        VALUES (%s, %s, %s, %s,
                                to_timestamp(%s/1000.0),
                                to_timestamp(%s/1000.0),
                                %s,
                                %s, %s, %s, %s)
                        ON CONFLICT (match_id) DO NOTHING;
                    """, (
                        match_id,
                        series_id,
                        match_desc,
                        match_format,
                        start_date,
                        end_date,
                        status,
                        team1_id,
                        team2_id,
                        winner_id,
                        venue
                    ))

                except Exception as e:
                    print(f"❌ Error inserting match {match_id}: {e}")
                    conn_db.rollback()  # 🔥 critical

    conn_db.commit()
    cursor.close()
    conn_db.close()

    print("✅ Matches inserted successfully!")


# ▶️ RUN
fetch_and_insert_matches()

✅ Matches inserted successfully!


Stadium Venues Insertion


In [237]:
import http.client
import json
import pandas as pd

In [238]:
def fetch_venue_data(venue_id):
    conn = http.client.HTTPSConnection("cricbuzz-cricket.p.rapidapi.com")

    headers = {
     'x-rapidapi-key': "a047d03cacmsh2a080d74736faf7p1f3118jsn380b850bb1d0",
    'x-rapidapi-host': "cricbuzz-cricket.p.rapidapi.com",
    }

    endpoint = f"/venues/v1/{venue_id}"
    conn.request("GET", endpoint, headers=headers)

    res = conn.getresponse()
    data = res.read()

    venue = json.loads(data.decode("utf-8"))
    return venue

In [239]:
venue = fetch_venue_data(45)
venue

{'ground': 'Basin Reserve',
 'city': 'Wellington',
 'country': 'New Zealand',
 'timezone': '+13:00',
 'capacity': '11,600',
 'ends': 'Vance Stand End, Scoreboard End',
 'homeTeam': 'Wellington',
 'imageUrl': 'http://i.cricketcb.com/i/stats/fth/540x303/venue/images/45.jpg',
 'imageId': '189174'}

In [240]:
import http.client
import json

def fetch_matches():
    conn = http.client.HTTPSConnection("cricbuzz-cricket.p.rapidapi.com")

    headers = {
         'x-rapidapi-key': "a047d03cacmsh2a080d74736faf7p1f3118jsn380b850bb1d0",
    'x-rapidapi-host': "cricbuzz-cricket.p.rapidapi.com",
    }

    # This endpoint gives list of matches
    conn.request("GET", "/matches/v1/recent", headers=headers)

    res = conn.getresponse()
    data = res.read()

    return json.loads(data.decode("utf-8"))

In [241]:
def extract_venue_ids(match_data):
    venue_ids = set()

    for match_type in match_data.get("typeMatches", []):
        for series in match_type.get("seriesMatches", []):
            for match in series.get("seriesAdWrapper", {}).get("matches", []):
                
                info = match.get("matchInfo", {})
                venue = info.get("venueInfo", {})

                vid = venue.get("id")
                if vid:
                    venue_ids.add(vid)

    return list(venue_ids)

In [242]:
matches = fetch_matches()
venue_ids = extract_venue_ids(matches)

venue_ids

[26,
 27,
 31,
 418,
 45,
 1461,
 1864,
 1482,
 1871,
 81,
 851,
 1438295,
 88,
 96,
 100,
 485,
 1893,
 363,
 371,
 116,
 122,
 380]

In [243]:
import re

def get_venue(venue_id, venue):
    raw_capacity = venue.get("capacity", "")

    # 🔥 Extract only numbers using regex
    numbers = re.findall(r"\d+", raw_capacity.replace(",", ""))

    capacity = int(numbers[0]) if numbers else None

    return {
        "venue_id": venue_id,
        "venue_name": venue.get("ground"),
        "city": venue.get("city"),
        "country": venue.get("country"),
        "timezone": venue.get("timezone"),
        "capacity": capacity,
        "ends": venue.get("ends"),
        "home_team": venue.get("homeTeam")
    }

In [244]:
def insert_venue(data):
    conn = get_connection()
    cursor = conn.cursor()

    query = """
        INSERT INTO venues (
            venue_id, venue_name, city, country, timezone,
            capacity, ends, home_team
        )
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
        ON CONFLICT (venue_id) DO NOTHING;
    """

    cursor.execute(query, (
        data["venue_id"],
        data["venue_name"],
        data["city"],
        data["country"],
        data["timezone"],
        data["capacity"],
        data["ends"],
        data["home_team"]
    ))

    conn.commit()
    cursor.close()
    conn.close()

In [245]:
def fetch_store_from_matches():
    matches = fetch_matches()
    venue_ids = extract_venue_ids(matches)

    print(f"Found {len(venue_ids)} venues")

    for vid in venue_ids:
        try:
            venue = fetch_venue_data(vid)
            processed = get_venue(vid, venue)
            insert_venue(processed)

            print(f"✅ Stored venue {vid}")

        except Exception as e:
            print(f"⚠️ Skipped {vid}: {e}")

Extracting Recent Matches


In [246]:
def extract_matches_and_teams(data):
    matches = []
    teams = {}

    for match_type in data.get("typeMatches", []):
        for series in match_type.get("seriesMatches", []):
            wrapper = series.get("seriesAdWrapper")
            if not wrapper:
                continue

            for match in wrapper.get("matches", []):
                info = match.get("matchInfo", {})

                # -----------------------
                # Extract Teams
                # -----------------------
                t1 = info.get("team1", {})
                t2 = info.get("team2", {})

                if t1:
                    teams[t1["teamId"]] = {
                        "team_id": t1["teamId"],
                        "team_name": t1["teamName"],
                        "short_name": t1["teamSName"]
                    }

                if t2:
                    teams[t2["teamId"]] = {
                        "team_id": t2["teamId"],
                        "team_name": t2["teamName"],
                        "short_name": t2["teamSName"]
                    }

                # -----------------------
                # Convert timestamp
                # -----------------------
                from datetime import datetime
                start_ts = info.get("startDate")
                start_date = datetime.fromtimestamp(int(start_ts)/1000) if start_ts else None

                # -----------------------
                # Extract Match
                # -----------------------
                matches.append({
                    "match_id": info.get("matchId"),
                    "series_id": info.get("seriesId"),
                    "match_desc": info.get("matchDesc"),
                    "match_format": info.get("matchFormat"),
                    "start_date": start_date,
                    "team1_id": t1.get("teamId"),
                    "team2_id": t2.get("teamId"),
                    "venue_id": info.get("venueInfo", {}).get("id")
                })

    return matches, list(teams.values())

In [247]:
data = fetch_matches()

matches, teams = extract_matches_and_teams(data)

print("Matches:", len(matches))
print("Teams:", len(teams))

Matches: 41
Teams: 51


In [248]:
def insert_teams(teams):
    conn = get_connection()
    cursor = conn.cursor()

    query = """
    INSERT INTO teams (team_id, team_name, short_name)
    VALUES (%s, %s, %s)
    ON CONFLICT (team_id) DO NOTHING;
    """

    for t in teams:
        cursor.execute(query, (
            t["team_id"],
            t["team_name"],
            t["short_name"]
        ))

    conn.commit()
    cursor.close()
    conn.close()

In [249]:
from datetime import datetime

def insert_matches(matches):
    conn = get_connection()
    cursor = conn.cursor()

    query = """
    INSERT INTO matches (
        match_id, series_id, match_desc, match_format,
        start_date, end_date, status,
        team1_id, team2_id, venue_id
    )
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    ON CONFLICT (match_id) DO NOTHING;
    """

    for m in matches:
        cursor.execute(query, (
            m["match_id"],
            m["series_id"],
            m["match_desc"],
            m["match_format"],
            m["start_date"],
            m.get("end_date"),
            m.get("status"),
            m["team1_id"],
            m["team2_id"],
            m["venue_id"]
        ))

    conn.commit()
    cursor.close()
    conn.close()

In [250]:
def extract_venue_ids_from_matches(matches):
    return list(set([m["venue_id"] for m in matches if m["venue_id"]]))

In [251]:
def insert_missing_venues(matches):
    venue_ids = extract_venue_ids_from_matches(matches)

    print(f"Found {len(venue_ids)} venues")

    for vid in venue_ids:
        try:
            venue = fetch_venue_data(vid)

            if not venue or "ground" not in venue:
                continue

            processed = get_venue(vid, venue)
            insert_venue(processed)

            print(f"✅ Venue {vid} inserted")

        except Exception as e:
            print(f"⚠️ Skipped venue {vid}: {e}")

In [252]:
data = fetch_matches()

matches, teams = extract_matches_and_teams(data)

# 🔥 FIX ORDER
insert_missing_venues(matches)   
insert_teams(teams)              
insert_matches(matches)          

print("✅ Full pipeline executed")

Found 22 venues
✅ Venue 26 inserted
✅ Venue 27 inserted
✅ Venue 31 inserted
✅ Venue 418 inserted
✅ Venue 45 inserted
✅ Venue 1461 inserted
✅ Venue 1864 inserted
✅ Venue 1482 inserted
✅ Venue 1871 inserted
✅ Venue 81 inserted
✅ Venue 851 inserted
✅ Venue 1438295 inserted
✅ Venue 88 inserted
✅ Venue 96 inserted
✅ Venue 100 inserted
✅ Venue 485 inserted
✅ Venue 1893 inserted
✅ Venue 363 inserted
✅ Venue 371 inserted
✅ Venue 116 inserted
✅ Venue 122 inserted
✅ Venue 380 inserted
✅ Full pipeline executed


Fetching data of top ODI run scorers

In [261]:
def fetch_top_run_scorers():
    conn = http.client.HTTPSConnection("cricbuzz-cricket.p.rapidapi.com")

    headers = {
          'x-rapidapi-key': "a047d03cacmsh2a080d74736faf7p1f3118jsn380b850bb1d0",
    'x-rapidapi-host': "cricbuzz-cricket.p.rapidapi.com",
    }

    conn.request(
        "GET",
        "/stats/v1/topstats/0?statsType=mostRuns&matchType=2",
        headers=headers
    )

    res = conn.getresponse()
    data = res.read()
    print(json.loads(data.decode("utf-8")))
    return json.loads(data.decode("utf-8"))

In [254]:
players = process_run_scorers(data)
print(players[:3])

[]


In [255]:
def process_run_scorers(data):
    players = []

    for row in data.get("values", []):
        vals = row["values"]

        players.append({
            "player_name": vals[1],
            "format_type": "ODI",
            "matches": int(vals[2]),
            "innings": int(vals[3]),
            "runs": int(vals[4]),
            "batting_average": float(vals[5]),
            "rank_position": int(vals[0])
        })

    return players

In [256]:
def insert_run_scorers(players):
    conn = get_connection()
    cursor = conn.cursor()

    query = """
    INSERT INTO top_run_scorers (
        player_name, format_type, matches, innings,
        runs, batting_average, rank_position
    )
    VALUES (%s, %s, %s, %s, %s, %s, %s)
    ON CONFLICT (player_name, format_type) DO UPDATE SET
        runs = EXCLUDED.runs,
        batting_average = EXCLUDED.batting_average,
        rank_position = EXCLUDED.rank_position;
    """

    for p in players:
        cursor.execute(query, (
            p["player_name"],
            p["format_type"],
            p["matches"],
            p["innings"],
            p["runs"],
            p["batting_average"],
            p["rank_position"]
        ))

    conn.commit()
    cursor.close()
    conn.close()

In [257]:
data = fetch_top_run_scorers()
players = process_run_scorers(data)

insert_run_scorers(players)

print("✅ Run scorers inserted")

✅ Run scorers inserted


In [258]:
data = fetch_top_run_scorers()
print(data)

{'filter': {'matchtype': [{'matchTypeId': '1', 'matchTypeDesc': 'test'}, {'matchTypeId': '2', 'matchTypeDesc': 'odi'}, {'matchTypeId': '3', 'matchTypeDesc': 't20i'}], 'team': [{'id': '2', 'teamShortName': 'IND'}, {'id': '27', 'teamShortName': 'IRE'}, {'id': '3', 'teamShortName': 'PAK'}, {'id': '4', 'teamShortName': 'AUS'}, {'id': '5', 'teamShortName': 'SL'}, {'id': '6', 'teamShortName': 'BAN'}, {'id': '9', 'teamShortName': 'ENG'}, {'id': '10', 'teamShortName': 'WI'}, {'id': '11', 'teamShortName': 'RSA'}, {'id': '12', 'teamShortName': 'ZIM'}, {'id': '13', 'teamShortName': 'NZ'}, {'id': '96', 'teamShortName': 'AFG'}], 'selectedMatchType': 'odi'}, 'headers': ['Batter', 'M', 'I', 'R', 'Avg'], 'values': [{'values': ['25', 'Tendulkar', '463', '452', '18426', '44.83']}, {'values': ['1413', 'Virat Kohli', '311', '299', '14797', '58.72']}, {'values': ['104', 'Sangakkara', '404', '380', '14234', '41.99']}, {'values': ['38', 'Ricky Ponting', '375', '365', '13704', '42.04']}, {'values': ['102', 'S

In [259]:
df = pd.read_sql("SELECT COUNT(*) FROM top_run_scorers;", get_connection())
df

C:\Users\hp\AppData\Local\Temp\ipykernel_34636\3334193213.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT COUNT(*) FROM top_run_scorers;", get_connection())


,count
0,20


Highest score in each format


In [290]:
import http.client

conn = http.client.HTTPSConnection("cricbuzz-cricket.p.rapidapi.com")

headers = {
    'x-rapidapi-key': "274cf7b009msh198245adc41e439p1f3633jsna216dc087842",
    'x-rapidapi-host': "cricbuzz-cricket.p.rapidapi.com",
    'Content-Type': "application/json"
}

conn.request("GET", "/stats/v1/topstats?statsType=highestScore&formatType=t20", headers=headers)

res = conn.getresponse()
data = res.read()

print(data.decode("utf-8"))

{"filter":{"matchtype":[{"matchTypeId":"1","matchTypeDesc":"test"},{"matchTypeId":"2","matchTypeDesc":"odi"},{"matchTypeId":"3","matchTypeDesc":"t20i"}],"team":[{"id":"2","teamShortName":"IND"},{"id":"27","teamShortName":"IRE"},{"id":"3","teamShortName":"PAK"},{"id":"4","teamShortName":"AUS"},{"id":"5","teamShortName":"SL"},{"id":"6","teamShortName":"BAN"},{"id":"9","teamShortName":"ENG"},{"id":"10","teamShortName":"WI"},{"id":"11","teamShortName":"RSA"},{"id":"12","teamShortName":"ZIM"},{"id":"13","teamShortName":"NZ"},{"id":"96","teamShortName":"AFG"}],"selectedMatchType":"test"},"headers":["Batter","HS","Balls","SR","Vs"],"values":[{"values":["240","B Lara","400","582","68.73","ENG"]},{"values":["157","Matthew Hayden","380","437","86.96","ZIM"]},{"values":["240","B Lara","375","538","69.70","ENG"]},{"values":["101","Mahela","374","572","65.38","RSA"]},{"values":["11200","Mulder","367","334","109.88","ZIM"]},{"values":["4160","G Sobers","365","","","PAK"]},{"values":["5141","S Hutton

In [ ]:
import http.client
import json


def get_highest_score_header():
    # 🔗 API connection
    conn = http.client.HTTPSConnection("cricbuzz-cricket.p.rapidapi.com")

    headers = {
        'x-rapidapi-key': "274cf7b009msh198245adc41e439p1f3633jsna216dc087842",
        'x-rapidapi-host': "cricbuzz-cricket.p.rapidapi.com",
        'Content-Type': "application/json"
    }

    try:
        # 📡 API request
        conn.request("GET", "/stats/v1/topstats", headers=headers)

        res = conn.getresponse()

        print(f"Status: {res.status}")

        if res.status != 200:
            print("❌ API failed")
            return

        # 📦 Read + parse JSON
        data = json.loads(res.read().decode("utf-8"))

        # 🔍 Extract "Highest Scores"
        for category in data.get("statsTypesList", []):
            for stat in category.get("types", []):
                if stat.get("value") == "highestScore":
                    print("✅ Header Found:", stat.get("header"))
                    return stat.get("header")

        print("⚠️ highestScore not found")

    except Exception as e:
        print(f"❌ Error: {e}")

    finally:
        conn.close()


# ▶️ RUN
if __name__ == "__main__":
    print("🚀 Running script...")
    result = get_highest_score_header()
    print("Final Output:", result)

{"statsTypesList":[{"types":[{"value":"mostRuns","header":"Most Runs","category":"Batting"},{"value":"highestScore","header":"Highest Scores","category":"Batting"},{"value":"highestAvg","header":"Best Batting Average","category":"Batting"},{"value":"highestSr","header":"Best Batting Strike Rate","category":"Batting"},{"value":"mostHundreds","header":"Most Hundreds","category":"Batting"},{"value":"mostFifties","header":"Most Fifties","category":"Batting"},{"value":"mostFours","header":"Most Fours","category":"Batting"},{"value":"mostSixes","header":"Most Sixes","category":"Batting"},{"value":"mostNineties","header":"Most Nineties","category":"Batting"}],"category":"Batting"},{"types":[{"value":"mostWickets","header":"Most Wickets","category":"Bowling"},{"value":"lowestAvg","header":"Best Bowling Average","category":"Bowling"},{"value":"bestBowlingInnings","header":"Best Bowling ","category":"Bowling"},{"value":"mostFiveWickets","header":"Most 5 Wickets Haul","category":"Bowling"},{"valu

In [302]:
def insert_highest_scores():
    import http.client
    import json

    print("🔥 FUNCTION STARTED")

    conn_db = get_connection()
    if not conn_db:
        print("❌ DB connection failed")
        return

    cursor = conn_db.cursor()

    headers = {
        'x-rapidapi-key': "274cf7b009msh198245adc41e439p1f3633jsna216dc087842",
        'x-rapidapi-host': "cricbuzz-cricket.p.rapidapi.com"
    }

    formats = ["test", "odi", "t20i"]

    for format_type in formats:
        print(f"\n📡 Fetching {format_type.upper()} data...")

        try:
            conn_api = http.client.HTTPSConnection("cricbuzz-cricket.p.rapidapi.com")

            conn_api.request(
                "GET",
                f"/stats/v1/topstats?statsType=highestScore&formatType={format_type}",
                headers=headers
            )

            res = conn_api.getresponse()
            print(f"👉 STATUS: {res.status}")

            if res.status != 200:
                print(f"❌ API failed for {format_type}")
                continue

            data = json.loads(res.read().decode("utf-8"))

            inserted_count = 0

            # 🔥 CORRECT PARSING (nested structure)
            for item in data.get("values", []):

                try:
                    row = item.get("values", [])

                    if not row or len(row) < 3:
                        continue

                    player_name = row[1]

                    # safe score parsing
                    try:
                        highest_score = int(row[2])
                    except:
                        continue

                    balls = int(row[3]) if len(row) > 3 and row[3].isdigit() else None

                    strike_rate = None
                    if len(row) > 4 and row[4]:
                        try:
                            strike_rate = float(row[4])
                        except:
                            pass

                    opponent = row[5] if len(row) > 5 else None

                    cursor.execute("""
                        INSERT INTO highest_scores 
                        (player_name, format_type, highest_score, balls, strike_rate, opponent)
                        VALUES (%s, %s, %s, %s, %s, %s)
                        ON CONFLICT DO NOTHING;
                    """, (
                        player_name,
                        format_type.upper(),
                        highest_score,
                        balls,
                        strike_rate,
                        opponent
                    ))

                    inserted_count += 1

                except Exception as e:
                    print(f"❌ Row error: {row} → {e}")
                    conn_db.rollback()

            conn_db.commit()
            print(f"✅ Inserted {inserted_count} rows for {format_type}")

        except Exception as e:
            print(f"❌ ERROR for {format_type}: {e}")

    cursor.close()
    conn_db.close()
    print("\n🎉 DONE inserting highest scores!")

Series from 2024


In [ ]:
import http.client
import json


def insert_series():
    conn_db = get_connection()
    if not conn_db:
        return

    cursor = conn_db.cursor()

    conn_api = http.client.HTTPSConnection("cricbuzz-cricket.p.rapidapi.com")

    headers = {
          'x-rapidapi-key': "274cf7b009msh198245adc41e439p1f3633jsna216dc087842",
    'x-rapidapi-host': "cricbuzz-cricket.p.rapidapi.com",
    }

    try:
        conn_api.request("GET", "/series/v1/archives/international", headers=headers)

        res = conn_api.getresponse()

        print("STATUS:", res.status)

        if res.status != 200:
            print("❌ API failed")
            return

        data = json.loads(res.read().decode("utf-8"))

        inserted = 0

        for year_block in data.get("seriesMapProto", []):
            year = year_block.get("date")

            for series in year_block.get("series", []):

                try:
                    series_id = series.get("id")
                    series_name = series.get("name")

                    start_dt = series.get("startDt")
                    end_dt = series.get("endDt")

                    cursor.execute("""
                        INSERT INTO series (
                            series_id,
                            series_name,
                            start_date,
                            end_date
                        )
                        VALUES (%s, %s,
                                to_timestamp(%s/1000.0),
                                to_timestamp(%s/1000.0))
                        ON CONFLICT DO NOTHING;
                    """, (
                        series_id,
                        series_name,
                        start_dt,
                        end_dt
                    ))

                    inserted += 1

                except Exception as e:
                    print(f"❌ Error: {e}")
                    conn_db.rollback()

        conn_db.commit()
        print(f"✅ Inserted {inserted} series")

    except Exception as e:
        print(f"❌ API error: {e}")

    finally:
        cursor.close()
        conn_db.close()


# ▶️ RUN
insert_series()

STATUS: 200
✅ Inserted 20 series


Most wickets and Runs

In [319]:
import http.client

conn = http.client.HTTPSConnection("cricbuzz-cricket.p.rapidapi.com")

headers = {
    'x-rapidapi-key': "274cf7b009msh198245adc41e439p1f3633jsna216dc087842",
    'x-rapidapi-host': "cricbuzz-cricket.p.rapidapi.com",
    'Content-Type': "application/json"
}

conn.request("GET", "/stats/v1/topstats/0?statsType=mostRuns&matchTypeDesc=odi", headers=headers)

res = conn.getresponse()
data = res.read()

print(data.decode("utf-8"))

{"filter":{"matchtype":[{"matchTypeId":"1","matchTypeDesc":"test"},{"matchTypeId":"2","matchTypeDesc":"odi"},{"matchTypeId":"3","matchTypeDesc":"t20i"}],"team":[{"id":"2","teamShortName":"IND"},{"id":"27","teamShortName":"IRE"},{"id":"3","teamShortName":"PAK"},{"id":"4","teamShortName":"AUS"},{"id":"5","teamShortName":"SL"},{"id":"6","teamShortName":"BAN"},{"id":"9","teamShortName":"ENG"},{"id":"10","teamShortName":"WI"},{"id":"11","teamShortName":"RSA"},{"id":"12","teamShortName":"ZIM"},{"id":"13","teamShortName":"NZ"},{"id":"96","teamShortName":"AFG"}],"selectedMatchType":"test"},"headers":["Batter","M","I","R","Avg"],"values":[{"values":["25","Tendulkar","200","329","15921","53.79"]},{"values":["8019","Root","163","298","13943","51.07"]},{"values":["38","Ricky Ponting","168","287","13378","51.85"]},{"values":["213","Kallis","166","280","13289","55.37"]},{"values":["27","Dravid","164","286","13288","52.31"]},{"values":["488","Cook","161","291","12472","45.35"]},{"values":["104","Sang